========================================
### GOLD LAYER – FACT ORDERS
----------------------------------------
###Source : Silver (Delta Tables)
###Target : Delta Gold Fact Table
###Grain  : One row per order
###Load   : Full Refresh (Overwrite)
###Purpose: Business analytics & reporting
========================================


### READ SILVER TABLE

In [0]:
df_orders = spark.table("olist_silver_orders")
df_items = spark.table("olist_silver_order_items")



### AGGREGATE



In [0]:
df_orders = spark.table("olist_silver_orders")
df_items = spark.table("olist_silver_order_items")


In [0]:
from pyspark.sql.functions import sum, count

df_items_agg = (
    df_items
    .groupBy("order_id")
    .agg(
        sum("price").alias("total_item_price"),
        sum("freight_value").alias("total_freight_value"),
        count("*").alias("item_count")
    )
)


### JOIN

In [0]:
df_fact_orders = (
    df_orders
    .join(df_items_agg, on="order_id", how="left")
)


### ADDING COLUMS

In [0]:
from pyspark.sql.functions import to_date, col

df_fact_orders = (
    df_fact_orders
    .withColumn("order_date", to_date("order_purchase_timestamp"))
    .withColumn(
        "total_order_value",
        col("total_item_price") + col("total_freight_value")
    )
)


In [0]:
df_fact_orders_final = df_fact_orders.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_date",
    "total_item_price",
    "total_freight_value",
    "total_order_value",
    "item_count"
)


### WRITE GOLD TABLE

In [0]:
(
    df_fact_orders_final
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist_gold_fact_orders")
)


In [0]:
%sql
SELECT COUNT(*) FROM olist_gold_fact_orders;


count(1)
99441


In [0]:
%sql
SELECT *
FROM olist_gold_fact_orders
ORDER BY total_order_value DESC
LIMIT 10;


order_id,customer_id,order_status,order_date,total_item_price,total_freight_value,total_order_value,item_count
03caa2c082116e1d31e67e9ae3700499,1617b1357756262bfa56ab541c47bc16,delivered,2017-09-29,26880.0,448.15999999999997,27328.16,16
736e1922ae60d0d6a89247b851902527,ec5b2ba62e574342386871631fafd3fc,delivered,2018-07-15,14320.0,229.76,14549.76,8
0812eb902a67711a1cb742b3cdaa65ae,c6e2731c5b391845f6800c97401a43a9,delivered,2017-02-12,13470.0,388.62,13858.62,2
fefacc66af859508bf1a7934eab1e97f,f48d464a0baaea338cb25f816991ab1f,delivered,2018-07-25,13458.0,386.42,13844.42,2
f5136e38d1a14a4dbd87dff67da82701,3fd6777bbce08a352fddd04e4a7cc8f6,delivered,2017-05-24,12998.0,455.32,13453.32,2
2cc9089445046817a7539d90805e6e5a,05455dfa7cd02f13d132aa7a6a9729c6,delivered,2017-11-24,11869.2,293.88,12163.08,12
a96610ab360d42a2e5335a3998b4718a,df55c14d1476a9a3467f131269c2477f,delivered,2017-04-01,9598.0,302.68,9900.68,2
b4c4b76c642808cbe472a32b86cddc95,e0a2412720e9ea4f26c1ac985f6a7358,canceled,2018-07-12,9199.8,419.08,9618.88,4
199af31afc78c699f0dbf71fb178d4d4,24bbf5fd2f2e1b359ee7de94defc4a15,delivered,2017-04-18,9380.0,148.68,9528.68,2
8dbc85d1447242f3b127dda390d56e19,3d979689f636322c62418b6346b1c6d2,delivered,2018-06-22,9180.0,183.56,9363.56,2


### OPTIMIZE 

In [0]:
# The following command optimizes the Delta table 'olist_gold_fact_orders' by compacting small files into larger ones,
# improving query performance and resource efficiency.

spark.sql("OPTIMIZE olist_gold_fact_orders")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

### Z-ORDER

In [0]:
# This command optimizes the Delta table 'olist_gold_fact_orders' by compacting small files and clustering data by 'order_date' and 'customer_id' to improve query performance.
spark.sql("""
    OPTIMIZE olist_gold_fact_orders
    ZORDER BY (order_date, customer_id)
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
spark.sql("DESCRIBE DETAIL olist_gold_fact_orders")


DataFrame[format: string, id: string, name: string, description: string, location: string, createdAt: timestamp, lastModified: timestamp, partitionColumns: array<string>, clusteringColumns: array<string>, numFiles: bigint, sizeInBytes: bigint, properties: map<string,string>, minReaderVersion: int, minWriterVersion: int, tableFeatures: array<string>, statistics: map<string,bigint>, clusterByAuto: boolean]

In [0]:
display(spark.sql("DESCRIBE DETAIL olist_gold_fact_orders"))


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,498bc686-ec3c-4848-bc2e-4832e7731c1e,adf_new.default.olist_gold_fact_orders,null,abfss://unity-catalog-storage@dbstoragetic7vxegr5zes.dfs.core.windows.net/7405614582366842/__unitystorage/catalogs/4444e7c2-d2e1-4e77-b5c3-b026efd7d282/tables/ab507e7d-1f61-4270-b886-3f20517da4e7,2026-01-31T18:39:25.341Z,2026-01-31T18:46:38Z,List(),List(),1,4250448,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false
